# Figure 11.14: Mini-batch k-means reduces computation time by using small random subsets…


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


def rng_from_seed(seed):
    return np.random.default_rng(int(seed))

def blobs(seed=5, n=240, noise=0.50):
    rng = rng_from_seed(seed)
    centers = np.array([[-2.0, -1.5], [2.0, -1.0], [0.2, 2.1]])
    per = n // len(centers)
    X = np.vstack([c + noise * rng.normal(size=(per, 2)) for c in centers])
    while len(X) < n:
        c = centers[len(X) % len(centers)]
        X = np.vstack([X, c + noise * rng.normal(size=(1, 2))])
    return X


def init_from_data(X, k, seed):
    rng = rng_from_seed(seed)
    idx = rng.choice(len(X), size=k, replace=False)
    return X[idx].copy()


def assign_all(X, C):
    d = ((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)
    return d.argmin(axis=1)


def update_means(X, labels, C):
    out = []
    for j in range(len(C)):
        pts = X[labels == j]
        out.append(pts.mean(axis=0) if len(pts) else C[j])
    return np.vstack(out)


def inertia(X, C):
    labels = assign_all(X, C)
    return np.sum((X - C[labels]) ** 2)


def standard_kmeans(X, k, steps, seed):
    C = init_from_data(X, k, seed)
    hist = []
    for _ in range(steps):
        labels = assign_all(X, C)
        C = update_means(X, labels, C)
        hist.append(inertia(X, C))
    return C, assign_all(X, C), np.array(hist)


def minibatch_kmeans(X, k, batch, steps, seed):
    rng = rng_from_seed(seed + 77)
    C = init_from_data(X, k, seed)
    counts = np.zeros(k)
    hist = []
    for _ in range(steps):
        idx = rng.choice(len(X), size=batch, replace=False)
        xb = X[idx]
        labels = assign_all(xb, C)
        for p, j in zip(xb, labels):
            counts[j] += 1
            eta = 1.0 / counts[j]
            C[j] = (1 - eta) * C[j] + eta * p
        hist.append(inertia(X, C))
    return C, assign_all(X, C), np.array(hist)


def draw(k=3, batch_size=20, updates=30, noise=0.50):
    X = blobs(n=240, noise=noise)
    C1, y1, h1 = standard_kmeans(X, k, updates, seed=9)
    C2, y2, h2 = minibatch_kmeans(X, k, batch_size, updates, seed=9)
    fig, axs = plt.subplots(1, 2, figsize=(13.0, 4.5))
    axs[0].scatter(X[:, 0], X[:, 1], c=y1, s=16, cmap="tab10", alpha=0.7)
    axs[0].scatter(C1[:, 0], C1[:, 1], c="k", marker="x", s=170, linewidths=3, label="standard")
    axs[0].scatter(C2[:, 0], C2[:, 1], c="#dc2626", marker="D", s=70, label="mini-batch")
    axs[0].set_title("Final centroids")
    axs[0].legend()
    axs[0].grid(alpha=0.2)
    axs[1].plot(np.arange(1, updates + 1), h1, "-o", color="#2563eb", label="standard")
    axs[1].plot(np.arange(1, updates + 1), h2, "-o", color="#dc2626", label="mini-batch")
    axs[1].set_title("Optimization tradeoff")
    axs[1].set_xlabel("update")
    axs[1].set_ylabel("full-dataset inertia")
    axs[1].grid(alpha=0.2)
    axs[1].legend()
    plt.show()

widgets.interact(
    draw,
    k=widgets.IntSlider(min=2, max=6, step=1, value=3, continuous_update=False),
    batch_size=widgets.IntSlider(min=5, max=80, step=5, value=20, continuous_update=False),
    updates=widgets.IntSlider(min=5, max=80, step=5, value=30, continuous_update=False),
    noise=widgets.FloatSlider(min=0.20, max=1.20, step=0.05, value=0.50, continuous_update=False),
)
